In [19]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [20]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



Step 1: Setup & Quantum Random Bit Generator

In [21]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()

def random_bit():
    """Generate a truly random bit by measuring |+⟩"""
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(qc, shots=1).result()
    return int(list(result.get_counts().keys())[0])

def random_bits(n):
    return [random_bit() for _ in range(n)]

Step 2 - Alice's Side


In [22]:
N = 100  # number of qubits to send

alice_bits  = random_bits(N)   # the secret bits
alice_bases = random_bits(N)   # 0 = rectilinear (+), 1 = diagonal (×)

def encode_qubit(bit, basis):
    """Alice encodes one qubit"""
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)          # flip to |1⟩
    if basis == 1:
        qc.h(0)          # rotate to diagonal basis
    return qc

Step 3 - Bob's Side

In [23]:
bob_bases = random_bits(N)

def measure_qubit(qc, basis):
    """Bob measures in his chosen basis"""
    if basis == 1:
        qc.h(0)          # rotate back from diagonal
    qc.measure(0, 0)
    result = simulator.run(qc, shots=1).result()
    return int(list(result.get_counts().keys())[0])

bob_results = []
for i in range(N):
    qc = encode_qubit(alice_bits[i], alice_bases[i])
    bit = measure_qubit(qc, bob_bases[i])
    bob_results.append(bit)

Step 4 — Sifting (Classical Channel)

In [24]:
# === SIFTING ===
sifted_alice = []
sifted_bob   = []

for i in range(N):
    if alice_bases[i] == bob_bases[i]:   # bases matched
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

print(f"Sifted key length: {len(sifted_alice)} bits")

Sifted key length: 49 bits


Step 5 - Error Checking

In [25]:
sample_size = 10  # bits to sacrifice for checking
THRESHOLD   = 0.1

sample_alice = sifted_alice[:sample_size]
sample_bob   = sifted_bob[:sample_size]

errors = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate = errors / sample_size

print(f"Error rate: {error_rate:.0%}")

if error_rate > THRESHOLD:
    print("  Attack detected! Aborting.")
else:
    final_key = sifted_alice[sample_size:]  # remaining bits = shared key
    print(f" No attack detected. Key length: {len(final_key)} bits")
    print(f"Key: {final_key}")

Error rate: 0%
 No attack detected. Key length: 39 bits
Key: [1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1]
